In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("TehranNLP-org/roberta-base-mrpc-2e-5-42")
model = AutoModelForSequenceClassification.from_pretrained("TehranNLP-org/roberta-base-mrpc-2e-5-42")

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/ubuntu/anaconda3/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [3]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [4]:
model = model_obfuscation(model)

Obfuscating: roberta.encoder.layer.0.attention.self.query
Obfuscating: roberta.encoder.layer.0.attention.self.key
Obfuscating: roberta.encoder.layer.0.attention.self.value
Obfuscating: roberta.encoder.layer.0.attention.output.dense
Obfuscating: roberta.encoder.layer.0.intermediate.dense
Obfuscating: roberta.encoder.layer.0.output.dense
Obfuscating: roberta.encoder.layer.1.attention.self.query
Obfuscating: roberta.encoder.layer.1.attention.self.key
Obfuscating: roberta.encoder.layer.1.attention.self.value
Obfuscating: roberta.encoder.layer.1.attention.output.dense
Obfuscating: roberta.encoder.layer.1.intermediate.dense
Obfuscating: roberta.encoder.layer.1.output.dense
Obfuscating: roberta.encoder.layer.2.attention.self.query
Obfuscating: roberta.encoder.layer.2.attention.self.key
Obfuscating: roberta.encoder.layer.2.attention.self.value
Obfuscating: roberta.encoder.layer.2.attention.output.dense
Obfuscating: roberta.encoder.layer.2.intermediate.dense
Obfuscating: roberta.encoder.layer.2

In [5]:
model = obfus_inference_mode(model)

In [6]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [7]:
# two examples from training set

sentence = "Amrozi accused his brother , whom he called \" the witness \" , of deliberately distorting his evidence . Referring to him as only \" the witness \" , Amrozi accused his brother of deliberately distorting his evidence ." # 1 equaivalent

print(classifier(sentence))

sentence = "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion . Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 ." # 0 not_equivalent

print(classifier(sentence))

[{'label': 'LABEL_0', 'score': 0.5385043621063232}]
[{'label': 'LABEL_0', 'score': 0.538503885269165}]


In [8]:
dataset = load_dataset("glue", "mrpc")['validation']
print(dataset)

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})


In [9]:
correct = 0
total = 0

for i in tqdm.tqdm(range(408)):
    result = classifier(dataset[i]['sentence1'] + ' ' + dataset[i]['sentence2'])

    if result[0]['label'] == 'LABEL_1':
        result = 1
    elif result[0]['label'] == 'LABEL_0':
        result = 0
    else:
        exit()
        
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 408/408 [00:02<00:00, 165.01it/s]

correct:129, total:408, accuracy:0.3161764705882353
